# Projectors definitions — bipartite process matrices

This notebook validates the explicit projector block used by the repository.

Conventions:
- subsystem order: `[AI, AO, BI, BO]`;
- trace convention: `Tr(W)=d_O=d_AO*d_BO`;
- replacement map: `_S W = (I_S/d_S) ⊗ Tr_S(W)`.

The implemented valid-process subspace is the intersection of the standard bipartite linear constraints:

1. `[1-B_O] A_I A_O W = 0`,
2. `[1-A_O] B_I B_O W = 0`,
3. `[1-A_O][1-B_O] W = 0`.

Fixed-order projectors are implemented as no-future-output subspaces: `A≺B` uses `W=_B_O W`, and `B≺A` uses `W=_A_O W`.

In [ ]:
from pathlib import Path
import json
import numpy as np

from deltawkrel.projectors import (
    ProcessDims,
    L_valid_bipartite, L_A_before_B, L_B_before_A,
    trace_replace, superoperator_matrix, random_hermitian,
    white_noise_process, assert_trace_convention, is_psd
)
from deltawkrel.switch_models import (
    fixed_order_A_before_B_process, fixed_order_B_before_A_process, causally_separable_mixture
)

ROOT = Path('..') if Path('../src').exists() else Path('.')
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
dims = ProcessDims(2,2,2,2)
print(dims)

## 1. Idempotence of replacement and process projectors

In [ ]:
W = random_hermitian(dims, seed=123).real
projectors = {
    'L_V': L_valid_bipartite,
    'L_AB': L_A_before_B,
    'L_BA': L_B_before_A,
}
checks = {}
for name, P in projectors.items():
    PW = P(W, dims)
    err = float(np.linalg.norm(P(PW, dims) - PW))
    checks[f'{name}_idempotence_error'] = err
    print(name, err)
    assert err < 1e-8

# Replacement maps themselves are also idempotent.
for systems in ([1], [3], [1,3], [0,1], [2,3]):
    R = trace_replace(W, dims, systems)
    err = float(np.linalg.norm(trace_replace(R, dims, systems) - R))
    checks[f'replace_{systems}_idempotence_error'] = err
    assert err < 1e-8

## 2. Membership of white-noise and fixed-order validation targets

In [ ]:
targets = {
    'white_noise': white_noise_process(dims),
    'fixed_order_A_before_B': fixed_order_A_before_B_process(dims),
    'fixed_order_B_before_A': fixed_order_B_before_A_process(dims),
    'causally_separable_mixture_q037': causally_separable_mixture(dims, q=0.37),
}
for name, X in targets.items():
    assert_trace_convention(X, dims)
    assert is_psd(X), name
    valid_err = float(np.linalg.norm(L_valid_bipartite(X, dims) - X))
    checks[f'{name}_valid_membership_error'] = valid_err
    print(name, 'valid_err=', valid_err, 'trace=', np.trace(X))
    assert valid_err < 1e-8

assert np.linalg.norm(L_A_before_B(targets['fixed_order_A_before_B'], dims) - targets['fixed_order_A_before_B']) < 1e-8
assert np.linalg.norm(L_B_before_A(targets['fixed_order_B_before_A'], dims) - targets['fixed_order_B_before_A']) < 1e-8

## 3. Export vectorized projector matrices

These `.npz` matrices allow the SDP notebook and tests to reproduce the linear constraints without rebuilding them manually.

In [ ]:
P_V = superoperator_matrix(L_valid_bipartite, dims)
P_AB = superoperator_matrix(L_A_before_B, dims)
P_BA = superoperator_matrix(L_B_before_A, dims)
for name, P in [('P_V', P_V), ('P_AB', P_AB), ('P_BA', P_BA)]:
    err = float(np.linalg.norm(P @ P - P))
    checks[f'{name}_matrix_idempotence_error'] = err
    print(name, P.shape, 'idempotence=', err)
    assert err < 1e-8

np.savez_compressed(RESULTS / 'process_projectors_d2.npz', P_V=P_V, P_AB=P_AB, P_BA=P_BA)
(RESULTS / 'projectors_validation_report.json').write_text(
    json.dumps(checks, indent=2, allow_nan=False), encoding='utf-8'
)
print('Exported:', RESULTS / 'process_projectors_d2.npz')
print('Report:', RESULTS / 'projectors_validation_report.json')

## Status

This notebook now validates the concrete projector block for the bipartite `[AI, AO, BI, BO]` convention. It does **not** by itself validate the ideal quantum switch; that is handled, as a separate blocker, in `validation_switch_ideal.ipynb`.